In [ ]:
!foldseek

In [ ]:
import pandas as pd
from itertools import combinations
from src.foldseek import Foldseek

In [ ]:
structure_dir = "../local_snapshot/data/afcase"
structure_db = "../local_snapshot/data/folddb/folddb"
alignment_db = "../local_snapshot/data/alignment_db/alignment_db"
alignment_df_path = "../local_snapshot/data/dataframes/alignment_df.tsv"
alignment_matrix_path = "../local_snapshot/data/dataframes/alignment_matrix.csv"
temp_dir = "/tmp"

In [ ]:
foldseek = Foldseek()

In [ ]:
!mv ../local_snapshot/data/folddb* ../local_snapshot/data/folddb

In [ ]:
foldseek.createdb(input_file=structure_dir, output_file=structure_db)

In [ ]:
!foldseek createdb 

In [ ]:
!foldseek easy-search ../local_snapshot/data/folddb/folddb ../local_snapshot/data/folddb/folddb ../local_snapshot/data/alignment_db/alignment_db ../local_snapshot/data/tmp 

In [ ]:
foldseek createdb example/ db 
foldseek search db db aln tmpFolder -c 0.8 
foldseek clust db aln clu
foldseek createtsv db db clu clu.tsv

In [ ]:
!foldseek createdb ../local_snapshot/data/afcase ../local_snapshot/data/folddb/case

In [ ]:
!foldseek search ../local_snapshot/data/folddb/case ../local_snapshot/data/folddb/case ../local_snapshot/data/alignment_db/case ../local_snapshot/data/tmp -c 0.9

In [ ]:
!foldseek clust ../local_snapshot/data/folddb/case ../local_snapshot/data/alignment_db/case ../local_snapshot/data/clu/clu

In [ ]:
!foldseek createtsv ../local_snapshot/data/folddb/case ../local_snapshot/data/folddb/case ../local_snapshot/data/clu/clu ../local_snapshot/data/clu/clu.tsv

In [ ]:
!foldseek easy-cluster ../local_snapshot/data/afnew_foldseek_key ../local_snapshot/data/all/new_foldseek_key ../local_snapshot/data/tmp -c 0.9 

In [ ]:
foldseek.createtsv(query_db=structure_db, target_db=structure_db, alignment_db=alignment_db, output_path=alignment_df_path)

In [ ]:
!foldseek createtsv ../local_snapshot/data/folddb/folddb ../local_snapshot/data/folddb/folddb ../local_snapshot/data/alignment_db/alignment_db ../local_snapshot/data/dataframes/alignment_df.tsv


In [ ]:
!mmseqs easy-cluster ../local_snapshot/data/EC1.17.99.9.fasta '../local_snapshot/data/case/case_1.17.99.9' ../local_snapshot/data/tmp --min-seq-id 0.5 -c 0.9 --cov-mode 1

In [ ]:
!foldseek easy-cluster ../local_snapshot/data/af1.17.99.9 ../local_snapshot/data/case/EC_1.17.99.9 ../local_snapshot/data/tmp -c 0.9 

In [ ]:
!foldseek easy-cluster ../local_snapshot/data/afall_ID100 ../local_snapshot/data/all/afall_ID100 ../local_snapshot/data/tmp -c 0.9 

In [ ]:
alignment_df = pd.read_csv(alignment_df_path, delimiter="\t", header=None)
alignment_df

In [ ]:
alignment_df = alignment_df.iloc[:,[0,1,4]]
alignment_df.columns = ["query", "target", "tmscore"]
alignment_df

In [ ]:
combinations_list = list(combinations(alignment_df["query"].unique(), 2))
len(combinations_list)

In [ ]:
filtered_df = alignment_df[alignment_df.apply(lambda row: (row['query'], row['target']) in combinations_list, axis=1)]
filtered_df

In [ ]:
# Trim the ".pdb" part from the 'query' column
filtered_df['query'] = filtered_df['query'].str.replace('.pdb', '')
# Trim the ".pdb" part from the 'target' column
filtered_df['target'] = filtered_df['target'].str.replace('.pdb', '')

In [ ]:
alignment_matrix = filtered_df.pivot_table(columns="query", index="target", values="tmscore")

missing_col = [i for i in alignment_matrix.index if i not in alignment_matrix.columns]
missing_index = [i for i in alignment_matrix.columns if i not in alignment_matrix.index]

alignment_matrix.loc[missing_index[0]] = ["None" for i in range(len(alignment_matrix.columns))]
alignment_matrix[missing_col[0]] = ["None" for i in range(len(alignment_matrix.index))]

alignment_matrix.sort_index(inplace=True)
alignment_matrix.sort_index(axis=1, inplace=True)

for i in alignment_matrix.index:
    for c in alignment_matrix.columns:
        if i == c:
            alignment_matrix.loc[i, c] = 1.0
        if pd.isnull(alignment_matrix.loc[i, c]) or alignment_matrix.loc[i, c] == "None":
            alignment_matrix.loc[i, c] = alignment_matrix.loc[c, i]

alignment_matrix = alignment_matrix[alignment_matrix.columns].astype(float)
alignment_matrix = alignment_matrix.apply(lambda x: round(1 - x, 4))
alignment_matrix